In [1]:
import numpy as np
import networkx as nx
from scipy.special import logsumexp
from sklearn.metrics import normalized_mutual_info_score
import torch

# Bernoulli SBM

In [2]:
def binary_sbm_estimate(A, num_blocks, iter):
    gamma_prior = np.random.rand(num_blocks)
    gamma_prior = gamma_prior / np.sum(gamma_prior)
    
    beta = np.random.uniform(size = (num_blocks, num_blocks))

    gamma_posterior = np.random.rand(n, num_blocks)
    gamma_posterior = gamma_posterior / np.sum(gamma_posterior, axis=1, keepdims=True)
    print(gamma_posterior[0, :])
    for _ in range(iter):
        # E step
        gamma_posterior_temp = gamma_posterior.copy()
        log_gamma = np.zeros_like(gamma_posterior)
        for i in range(n):
            for k in range(num_blocks):
                term_1 = np.log(gamma_prior[k])
                term_2 = 0
                for j in range(n):
                    if j != i:
                        term_2 += np.sum(gamma_posterior[j, :]*(A[i, j]*np.log(beta[k, :]) + (1-A[i, j])*np.log(1-beta[k, :])))
                gamma_posterior_temp[i, k] = np.exp(term_1 + term_2)
                log_gamma[i, k] = term_1 + term_2
            log_gamma[i, :] = log_gamma[i, :] - logsumexp(log_gamma[i, :])
            gamma_posterior[i, :] = np.exp(log_gamma[i, :])




        ## Monte Carlo sample Z (community membership) -> to calculate the ELBO
        K = 1000
        samples = np.empty((n, K), dtype=np.int64)
        for i in range(n):
            samples[i] = np.random.choice(num_blocks, size=K, p=gamma_posterior[i])

        term_1 = 0        
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                else:
                    for l in range(num_blocks):
                        for k in range(num_blocks):
                            term_1 += gamma_posterior[i, l]*gamma_posterior[j, k]*(A[i, j]*np.log(beta[k, l]) + (1-A[i, j])*np.log(1-beta[k, l]))
        
        term_2 = 0
        for k in range(K):
            term_2 += np.sum(np.log(gamma_posterior[np.arange(n), samples[:, k]]))
        term_2 = term_2/K


        ELBO = term_1 + term_2
        print(f"ELBO: {ELBO}")

        # M step
        gamma_prior = np.average(gamma_posterior, axis=0)

        for k in range(num_blocks):
            for l in range(k, num_blocks):
                numerator = np.sum(np.outer(gamma_posterior[:,k],gamma_posterior[:,l]) * (1- np.eye(n)) * A)/2
                denominator = np.sum(np.outer(gamma_posterior[:,k],gamma_posterior[:,l]) * (1- np.eye(n)))/2
                beta[k, l] = numerator/denominator
                beta[l, k] = beta[k, l]

        print(gamma_prior)
        print(beta)
        # print(gamma_posterior)
    return gamma_posterior


In [10]:
seed = 0
# np.random.seed(0)


# Graph construction
block_sizes = [10,90]
num_blocks = len(block_sizes)

n = np.sum(block_sizes)
p = 1.5*n**(-0.3)
r = 0.15*n**(-0.3)
print(f"Intra prob: {p}")
print(f"Inter prob: {r}")

probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

G = nx.stochastic_block_model(block_sizes, probs, seed=seed)
A = nx.to_numpy_array(G)


gamma_posterior = binary_sbm_estimate(A, num_blocks, 20)


# NMI
block_labels = np.array([d['block'] for _, d in G.nodes(data=True)])
predicted_labels = np.argmax(gamma_posterior, axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")


Intra prob: 0.376782964726437
Inter prob: 0.0376782964726437
[0.33925578 0.66074422]
ELBO: -17201.654500247758
[1.0000000e+00 4.8956953e-14]
[[0.30848485 0.0586803 ]
 [0.0586803  0.14304607]]
ELBO: -6117.031891815662
[9.99995923e-01 4.07679578e-06]
[[0.30848702 0.04265029]
 [0.04265029 0.00772452]]
ELBO: -5871.112886669062
[0.93068305 0.06931695]
[[0.34787    0.04843582]
 [0.04843582 0.25469093]]
ELBO: -5708.825540131699
[0.90000766 0.09999234]
[[0.36578857 0.05666292]
 [0.05666292 0.24444785]]
ELBO: -5701.901091676607
[0.90000044 0.09999956]
[[0.36579252 0.05666645]
 [0.05666645 0.24444464]]
ELBO: -5701.901090354541
[0.90000044 0.09999956]
[[0.36579252 0.05666645]
 [0.05666645 0.24444464]]
ELBO: -5701.90109035454
[0.90000044 0.09999956]
[[0.36579252 0.05666645]
 [0.05666645 0.24444464]]
ELBO: -5701.901090354397
[0.90000044 0.09999956]
[[0.36579252 0.05666645]
 [0.05666645 0.24444464]]
ELBO: -5701.901090354397
[0.90000044 0.09999956]
[[0.36579252 0.05666645]
 [0.05666645 0.24444464]]
E

# Bernoulli SBM - Fully variational with random group proportion - SGD

In [233]:
import torch
import torch.nn.functional as F
from torch.special import digamma, gammaln

In [ ]:
def elbo_sampled(A, gamma_logits, log_alpha1, log_alpha2, log_rho, eta, sample_pct=0.1):
    N, K = gamma_logits.shape

    gamma  = F.softmax(gamma_logits, dim=-1)
    alpha1 = F.softplus(log_alpha1)
    alpha2 = F.softplus(log_alpha2)
    rho    = F.softplus(log_rho)                        # (K,), variational Dirichlet params

    alpha_sum  = alpha1 + alpha2
    E_log_b    = digamma(alpha1) - digamma(alpha_sum)
    E_log_1m_b = digamma(alpha2) - digamma(alpha_sum)

    rho_0      = rho.sum()
    E_log_pi   = digamma(rho) - digamma(rho_0)          # (K,), E_q[log pi_k]

    # --- sample random node pairs with i != j ---
    total_pairs = N * (N - 1)
    n_samples   = max(1, int(sample_pct * total_pairs))
    idx         = torch.randint(0, N, (n_samples * 2, 2))
    idx         = idx[idx[:, 0] != idx[:, 1]][:n_samples]
    i_idx, j_idx = idx[:, 0], idx[:, 1]
    n_pairs     = len(i_idx)
    scale       = total_pairs / n_pairs

    # 1. likelihood (scaled)
    gamma_i     = gamma[i_idx]
    gamma_j     = gamma[j_idx]
    A_ij        = A[i_idx, j_idx]
    gamma_outer = gamma_i.unsqueeze(2) * gamma_j.unsqueeze(1)   # (n_pairs, K, K)
    log_lik     = (A_ij.view(-1, 1, 1) * E_log_b
                   + (1 - A_ij).view(-1, 1, 1) * E_log_1m_b)
    L1 = scale * (gamma_outer * log_lik).sum()

    # 2. prior on Z: E_q[log p(Z|pi)] = sum_i sum_k gamma_ik * E_q[log pi_k]
    L2 = (gamma * E_log_pi.unsqueeze(0)).sum()

    # 3. prior on pi: E_q[log p(pi|eta)], Dirichlet(1,1,...) -> vanishes
#     eta_0 = eta.sum()
#     L3 = (gammaln(eta_0) - gammaln(eta).sum()
#           + ((eta - 1) * E_log_pi).sum())
    L3 = 0

    # 4. prior on beta: Beta(1,1) -> vanishes
    L4 = 0

    # 5. entropy of q_Z
    L5 = -(gamma * torch.log(gamma + 1e-10)).sum()

    # 6. entropy of q_pi (negative KL)
    L6 = -(gammaln(rho_0) - gammaln(rho).sum()
           - (rho_0 - K) * digamma(rho_0)
           + ((rho - 1) * digamma(rho)).sum())

    # 7. entropy of q_beta
    log_B = gammaln(alpha1) + gammaln(alpha2) - gammaln(alpha_sum)
    L7 = (log_B - (alpha1 - 1) * E_log_b - (alpha2 - 1) * E_log_1m_b).sum()

    return L1 + L2 + L3 + L4 + L5 + L6 + L7

In [273]:
def binary_sbm_estimate_fully_variational(
    A, num_blocks, iter, lr, lr_schedule=None, schedule_gamma=0.1, sample_pct=0.9
):
    N = A.shape[0]
    K = num_blocks

    gamma_logits = torch.randn(N, K).detach().requires_grad_(True)
    log_alpha1   = torch.zeros(K, K, requires_grad=True)
    log_alpha2   = torch.zeros(K, K, requires_grad=True)
    log_rho      = torch.zeros(K, requires_grad=True)

    eta = torch.ones(K)

    optimizer = torch.optim.Adam([gamma_logits, log_alpha1, log_alpha2, log_rho], lr=lr)

    for step in range(iter):
        if lr_schedule == "inverse":
            for pg in optimizer.param_groups:
                pg["lr"] = lr / (1.0 + schedule_gamma * step)
        optimizer.zero_grad()
        loss = -elbo_sampled(A, gamma_logits, log_alpha1, log_alpha2, log_rho, eta,
                             sample_pct=sample_pct)
        loss.backward()
        if step % 20 == 0:
            print(f"Step {step} loss: {loss.item():.4f}")
            if lr_schedule == "inverse":
                print(f"Learning rate: {lr / (1.0 + schedule_gamma * step)}")
            else:
                print(f"Learning rate: {optimizer.param_groups[0]['lr']}")
            print("gamma_logits.grad norm:", gamma_logits.grad.norm().item())
            # print("gamma_logits.grad:\n", gamma_logits.grad)
            print("log_alpha1.grad norm:", log_alpha1.grad.norm().item())
            # print("log_alpha1.grad:\n", log_alpha1.grad)
            print("log_alpha2.grad norm:", log_alpha2.grad.norm().item())
            # print("log_alpha2.grad:\n", log_alpha2.grad)
            print("log_rho.grad norm:", log_rho.grad.norm().item())
            # print("log_rho.grad:\n", log_rho.grad)
            print()
        optimizer.step()

    gamma_posterior = F.softmax(gamma_logits, dim=-1).detach()
    rho_posterior   = F.softplus(log_rho).detach()
    alpha1_posterior = F.softplus(log_alpha1).detach()
    alpha2_posterior = F.softplus(log_alpha2).detach()
    beta_posterior_mean = alpha1_posterior / (alpha1_posterior + alpha2_posterior)

    return gamma_posterior, rho_posterior, beta_posterior_mean

In [279]:
seed = 0
# np.random.seed(seed)
# torch.manual_seed(seed)

# Graph construction
block_sizes = [50,50]
num_blocks = len(block_sizes)

n = np.sum(block_sizes)
p = 1.5*n**(-0.3)
r = 0.15*n**(-0.3)
print(f"Intra prob: {p}")
print(f"Inter prob: {r}")

probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

G = nx.stochastic_block_model(block_sizes, probs, seed=seed)
A = nx.to_numpy_array(G)

gamma_posterior, rho_posterior, beta_posterior_mean = binary_sbm_estimate_fully_variational(
    torch.tensor(A, dtype=torch.float32), num_blocks, 10000, lr=3,
    lr_schedule="inverse", schedule_gamma=0.1, sample_pct=1
)

block_labels     = np.array([d['block'] for _, d in G.nodes(data=True)])
predicted_labels = np.argmax(gamma_posterior.numpy(), axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")
print(f"rho (posterior Dirichlet params): {rho_posterior.numpy()}")
print(f"pi (posterior mean): {(rho_posterior / rho_posterior.sum()).numpy()}")
print(f"\nbeta posterior mean (E[beta_kl]):")
print(np.round(beta_posterior_mean.numpy(), 3))

Intra prob: 0.376782964726437
Inter prob: 0.0376782964726437
Step 0 loss: 11585.0059
Learning rate: 3.0
gamma_logits.grad norm: 2.2749686241149902
log_alpha1.grad norm: 1150.2564697265625
log_alpha2.grad norm: 3135.630126953125
log_rho.grad norm: 28.783863067626953

Step 20 loss: 5878.1777
Learning rate: 1.0
gamma_logits.grad norm: 0.5312346816062927
log_alpha1.grad norm: 128.69015502929688
log_alpha2.grad norm: 85.77629089355469
log_rho.grad norm: 0.4737160801887512

Step 40 loss: 5600.2651
Learning rate: 0.6
gamma_logits.grad norm: 0.9624918699264526
log_alpha1.grad norm: 93.7710952758789
log_alpha2.grad norm: 53.88129425048828
log_rho.grad norm: 0.08767865598201752

Step 60 loss: 5433.8687
Learning rate: 0.42857142857142855
gamma_logits.grad norm: 0.5022187829017639
log_alpha1.grad norm: 75.7625503540039
log_alpha2.grad norm: 39.244144439697266
log_rho.grad norm: 0.17587639391422272

Step 80 loss: 5405.1797
Learning rate: 0.3333333333333333
gamma_logits.grad norm: 0.3847082853317261

In [97]:
gamma_posterior

tensor([[1.0000e+00, 4.2847e-12],
        [3.6631e-12, 1.0000e+00],
        [1.6154e-12, 1.0000e+00],
        [1.2371e-12, 1.0000e+00],
        [2.6006e-12, 1.0000e+00],
        [1.8025e-12, 1.0000e+00],
        [2.9226e-12, 1.0000e+00],
        [4.6200e-08, 1.0000e+00],
        [1.0011e-09, 1.0000e+00],
        [5.7583e-07, 1.0000e+00],
        [1.3626e-08, 1.0000e+00],
        [6.2932e-17, 1.0000e+00],
        [1.2549e-11, 1.0000e+00],
        [1.0000e+00, 7.5904e-10],
        [1.7027e-07, 1.0000e+00],
        [6.6276e-09, 1.0000e+00],
        [1.0000e+00, 3.1033e-08],
        [1.0000e+00, 4.2001e-11],
        [1.2205e-12, 1.0000e+00],
        [3.2319e-12, 1.0000e+00],
        [2.0573e-12, 1.0000e+00],
        [1.1306e-07, 1.0000e+00],
        [2.0964e-11, 1.0000e+00],
        [1.2868e-12, 1.0000e+00],
        [4.1202e-09, 1.0000e+00],
        [6.7756e-07, 1.0000e+00],
        [1.5453e-12, 1.0000e+00],
        [1.0000e+00, 4.4662e-12],
        [1.0000e+00, 2.6956e-12],
        [3.377

# Bernoulli SBM - Fully variational - SGD

In [3]:
import torch
import torch.nn.functional as F
from torch.special import digamma, gammaln

In [4]:
def elbo_sampled(A, gamma_logits, log_alpha1, log_alpha2, sample_pct=0.1):
    N, K = gamma_logits.shape

    gamma  = F.softmax(gamma_logits, dim=-1)
    alpha1 = F.softplus(log_alpha1)
    alpha2 = F.softplus(log_alpha2)

    alpha_sum  = alpha1 + alpha2
    E_log_b    = digamma(alpha1) - digamma(alpha_sum)
    E_log_1m_b = digamma(alpha2) - digamma(alpha_sum)

    # --- sample random node pairs with i != j ---
    total_pairs = N * (N - 1)
    n_samples   = max(1, int(sample_pct * total_pairs))

    idx    = torch.randint(0, N, (n_samples * 2, 2))
    idx    = idx[idx[:, 0] != idx[:, 1]][:n_samples]
    i_idx, j_idx = idx[:, 0], idx[:, 1]

    n_pairs = len(i_idx)
    scale   = total_pairs / n_pairs                 # unbiased scaling

    # 1. likelihood (scaled)
    gamma_i = gamma[i_idx]                          # (n_pairs, K)
    gamma_j = gamma[j_idx]                          # (n_pairs, K)
    A_ij    = A[i_idx, j_idx]                       # (n_pairs,)

    gamma_outer = (gamma_i.unsqueeze(2) *
                   gamma_j.unsqueeze(1))             # (n_pairs, K, K)
    log_lik     = (A_ij.view(-1, 1, 1) * E_log_b
                   + (1 - A_ij).view(-1, 1, 1) * E_log_1m_b)

    L1 = scale * (gamma_outer * log_lik).sum()

    # 2. prior on Z
    L2 = -N * torch.log(torch.tensor(float(K), dtype=torch.float32))

    # 4. entropy of q_Z
    L4 = -(gamma * torch.log(gamma + 1e-10)).sum()

    # 5. entropy of q_beta
    log_B = gammaln(alpha1) + gammaln(alpha2) - gammaln(alpha_sum)
    L5 = (log_B - (alpha1 - 1) * E_log_b - (alpha2 - 1) * E_log_1m_b).sum()

    return L1 + L2 + L4 + L5

In [5]:
def binary_sbm_estimate_fully_variational(
    A, num_blocks, iter, lr, lr_schedule=None, schedule_gamma=0.1, sample_pct=0.9
):
    """
    A: adjacency matrix
    lr_schedule: None (constant lr) or "inverse" for lr / (1 + schedule_gamma * step),
        same form as binary_sbm_estimate_E_step_GD.
    """
    N = A.shape[0]
    K = num_blocks

    gamma_logits = (torch.randn(N, K) * 1).detach().requires_grad_(True)
    log_alpha1   = torch.zeros(K, K, requires_grad=True)
    log_alpha2   = torch.zeros(K, K, requires_grad=True)

    optimizer = torch.optim.Adam([gamma_logits, log_alpha1, log_alpha2], lr=lr)

    for step in range(iter):
        if lr_schedule == "inverse":
            lr_t = lr / (1.0 + schedule_gamma * step)
            for pg in optimizer.param_groups:
                pg["lr"] = lr_t
        optimizer.zero_grad()
        loss = -elbo_sampled(A, gamma_logits, log_alpha1, log_alpha2, sample_pct=sample_pct)
        loss.backward()
        optimizer.step()
        print(f"Step {step} loss: {loss.item()}")

    gamma_posterior = F.softmax(gamma_logits, dim=-1).detach().numpy()
    return gamma_posterior


In [62]:
seed = 0
# np.random.seed(seed)
# torch.manual_seed(seed)

# Graph construction
block_sizes = [100,100]
num_blocks = len(block_sizes)

n = np.sum(block_sizes)
p = 1.5*n**(-0.3)
r = 0.15*n**(-0.3)
print(f"Intra prob: {p}")
print(f"Inter prob: {r}")

probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

G = nx.stochastic_block_model(block_sizes, probs, seed=seed)
A = nx.to_numpy_array(G)


# Estimate
gamma_posterior = binary_sbm_estimate_fully_variational(
    torch.tensor(A), num_blocks, 5000, lr=1,
    lr_schedule="inverse",
    schedule_gamma=0.03,
    sample_pct=0.9
)

# NMI
block_labels = np.array([d['block'] for _, d in G.nodes(data=True)])
predicted_labels = np.argmax(gamma_posterior, axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")

Intra prob: 0.30604286600525543
Inter prob: 0.030604286600525544
Step 0 loss: 46360.861169668286
Step 1 loss: 33238.86103428922
Step 2 loss: 31586.49494535659
Step 3 loss: 26919.81089158146
Step 4 loss: 23917.93086734065
Step 5 loss: 23037.343072174634
Step 6 loss: 22698.73662285383
Step 7 loss: 22174.819038000165
Step 8 loss: 21690.191595760774
Step 9 loss: 21437.72195047015
Step 10 loss: 20902.640565318496
Step 11 loss: 20637.64486406664
Step 12 loss: 20570.043803794495
Step 13 loss: 20509.939343628776
Step 14 loss: 20390.11962830667
Step 15 loss: 20451.85540946715
Step 16 loss: 20291.17774424042
Step 17 loss: 19936.240584962612
Step 18 loss: 19922.2396894857
Step 19 loss: 19969.505996812848
Step 20 loss: 19896.17659875666
Step 21 loss: 20082.278428481346
Step 22 loss: 19730.108362737657
Step 23 loss: 19771.408591545856
Step 24 loss: 19811.561347253424
Step 25 loss: 19777.35278261924
Step 26 loss: 19499.85302955682
Step 27 loss: 19640.589991562578
Step 28 loss: 19722.4686453222
Step 

In [7]:
gamma_posterior

array([[2.4272035e-07, 9.9999976e-01],
       [2.2778850e-06, 9.9999774e-01],
       [1.6007779e-06, 9.9999845e-01],
       [1.8934595e-06, 9.9999809e-01],
       [1.2766035e-06, 9.9999869e-01],
       [1.5827711e-06, 9.9999845e-01],
       [1.9358733e-06, 9.9999809e-01],
       [2.6178186e-06, 9.9999738e-01],
       [2.2573149e-06, 9.9999774e-01],
       [2.3528544e-06, 9.9999762e-01],
       [1.5506248e-06, 9.9999845e-01],
       [2.0941459e-06, 9.9999785e-01],
       [1.3508460e-06, 9.9999869e-01],
       [2.5647234e-06, 9.9999738e-01],
       [2.2162781e-06, 9.9999774e-01],
       [1.2818811e-06, 9.9999869e-01],
       [1.2254412e-06, 9.9999881e-01],
       [2.4836640e-06, 9.9999750e-01],
       [1.7077640e-06, 9.9999833e-01],
       [2.7817514e-06, 9.9999726e-01],
       [1.8069295e-06, 9.9999821e-01],
       [4.1732116e-07, 9.9999952e-01],
       [1.7392871e-06, 9.9999821e-01],
       [1.5559561e-06, 9.9999845e-01],
       [1.7852498e-06, 9.9999821e-01],
       [2.8854663e-06, 9.

# Bernoulli SBM - SGD E-Step

In [8]:
# Vectorized version
def binary_sbm_estimate_E_step_GD(A, num_blocks, iter, lr, mini_batch_p_sample):
    """
    A: adjacency matrix
    num_blocks: number of blocks
    iter: number of iterations
    K: number of Monte Carlo samples
    lr: learning rate
    beta[k,l] = P(A_ij=1 | z_i=k, z_j=l); kept symmetric (init and M-step).
    """

    gamma_prior = np.random.rand(num_blocks)
    gamma_prior = gamma_prior / np.sum(gamma_prior)

    beta = np.random.uniform(size=(num_blocks, num_blocks))
    beta = (beta + beta.T) / 2.0
    # Unconstrained logits, gamma_posterior = softmax(logits) so gamma_posterior is a probability distribution
    logits = torch.randn(n, num_blocks, dtype=torch.double, requires_grad=True)
    A_torch = torch.as_tensor(A, dtype=torch.double, device=logits.device)

    for t in range(iter):
        # E step — gradient ascent on ELBO w.r.t. logits
        # Learning-rate schedule
        lr_t = lr / (1.0 + 0.1 * t)

        gamma_posterior_torch = torch.nn.functional.softmax(logits, dim=1)
        beta_torch = torch.as_tensor(beta, dtype=torch.double, device=logits.device)
        log_beta = torch.log(beta_torch.clamp(min=1e-15))
        log_one_minus_beta = torch.log((1.0 - beta_torch).clamp(min=1e-15))

        # Upper-triangle pairs (i < j)
        i_idx, j_idx = torch.triu_indices(n, n, offset=1, device=logits.device)
        g_i = gamma_posterior_torch[i_idx]
        g_j = gamma_posterior_torch[j_idx]
        a_ij = A_torch[i_idx, j_idx]
        log_bern = (
            a_ij[:, None, None] * log_beta[None, :, :]
            + (1.0 - a_ij)[:, None, None] * log_one_minus_beta[None, :, :]
        )
        pair_mask = torch.bernoulli(
            torch.full(
                (i_idx.shape[0],),
                mini_batch_p_sample,
                dtype=logits.dtype,
                device=logits.device,
            )
        )
        edge_terms = torch.einsum("ek,ekl,el->e", g_i, log_bern, g_j)
        term_1 = (edge_terms * pair_mask).sum()

        term_2 = (gamma_posterior_torch * torch.log(gamma_posterior_torch.clamp(min=1e-15))).sum()

        ELBO = term_1 - term_2
        print(f"ELBO: {ELBO}")

        ELBO.backward()
        print("grad_norm:", logits.grad.norm().item())

        with torch.no_grad():
            logits.add_(lr_t * logits.grad)
        logits.grad.zero_()
        gamma_posterior = torch.nn.functional.softmax(logits.detach(), dim=1).cpu().numpy()


        # M step
        gamma_prior = np.average(gamma_posterior, axis=0)
        # gamma_prior = block_sizes/np.sum(block_sizes)

        # beta = probs
        for k in range(num_blocks):
            for l in range(k, num_blocks):
                numerator = np.sum(np.outer(gamma_posterior[:,k],gamma_posterior[:,l]) * (1- np.eye(n)) * A)/2
                denominator = np.sum(np.outer(gamma_posterior[:,k],gamma_posterior[:,l]) * (1- np.eye(n)))/2
                beta[k, l] = numerator/denominator
                beta[l, k] = beta[k, l]

    return gamma_posterior, gamma_prior, beta


In [59]:
seed = 0
# np.random.seed(seed)
# torch.manual_seed(seed)

# Graph construction
block_sizes = [50,50]
num_blocks = len(block_sizes)

n = np.sum(block_sizes)
p = 1.5*n**(-0.3)
r = 0.15*n**(-0.3)
print(f"Intra prob: {p}")
print(f"Inter prob: {r}")

probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

G = nx.stochastic_block_model(block_sizes, probs, seed=seed)
A = nx.to_numpy_array(G)


# Estimate
gamma_posterior, gamma_prior, beta = binary_sbm_estimate_E_step_GD(A, num_blocks, 40, lr = 0.1, mini_batch_p_sample = 0.8)

# NMI
block_labels = np.array([d['block'] for _, d in G.nodes(data=True)])
predicted_labels = np.argmax(gamma_posterior, axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")


Intra prob: 0.376782964726437
Inter prob: 0.0376782964726437
ELBO: -2060.914771680404
grad_norm: 6.435706693351839
ELBO: -1916.4205267765208
grad_norm: 2.39712171657678
ELBO: -1904.0841143044681
grad_norm: 2.373507662602153
ELBO: -1920.8350141929104
grad_norm: 2.3565709069085083
ELBO: -1941.7972110123194
grad_norm: 2.322449337515328
ELBO: -1884.3340290440292
grad_norm: 2.3223015557390405
ELBO: -1910.6987466417975
grad_norm: 2.285930540404052
ELBO: -1896.3223808827586
grad_norm: 2.274088652826453
ELBO: -1891.799900788259
grad_norm: 2.249073515960657
ELBO: -1916.416222489827
grad_norm: 2.261701267054256
ELBO: -1890.348139440746
grad_norm: 2.229085436689063
ELBO: -1905.6780968374756
grad_norm: 2.22625630316405
ELBO: -1891.0298288283616
grad_norm: 2.2074349407975813
ELBO: -1948.713594654074
grad_norm: 2.1932925661568974
ELBO: -1918.9474906909209
grad_norm: 2.1794501257701167
ELBO: -1924.1459304197656
grad_norm: 2.1708804212960597
ELBO: -1922.2381196554406
grad_norm: 2.164243395304858
ELBO:

In [ ]:
# Not vectorized version
def binary_sbm_estimate_E_step_GD(A, num_blocks, iter, lr, mini_batch_p_sample):
    """
    A: adjacency matrix
    num_blocks: number of blocks
    iter: number of iterations
    K: number of Monte Carlo samples
    lr: learning rate
    """
    gamma_prior = np.random.rand(num_blocks)
    gamma_prior = gamma_prior / np.sum(gamma_prior)
    
    beta = np.random.uniform(size=(num_blocks, num_blocks))
    beta = (beta + beta.T) / 2.0
    # Unconstrained logits, gamma_posterior = softmax(logits) so gamma_posterior is a probability distribution
    logits = torch.randn(n, num_blocks, dtype=torch.double, requires_grad=True)
    A_torch = torch.as_tensor(A, dtype=torch.double, device=logits.device)

    for t in range(iter):
        # E step — gradient ascent on ELBO w.r.t. logits
        # Learning-rate schedule
        lr_t = lr / (1.0 + 0.1 * t)

        gamma_posterior_torch = torch.nn.functional.softmax(logits, dim=1)
        beta_torch = torch.as_tensor(beta, dtype=torch.double)

        term_1 = 0        
        for i in range(n):
            for j in range(i+1,n):
                is_sampled = bool(np.random.choice(2, size=1, p=[1 - mini_batch_p_sample, mini_batch_p_sample]))
                if not is_sampled:
                    continue
                elif i == j:
                    continue
                else:
                    for l in range(num_blocks):
                        for k in range(num_blocks):
                            term_1 += gamma_posterior_torch[i, k]*gamma_posterior_torch[j, l]*(A_torch[i, j]*torch.log(beta_torch[k, l]) + (1-A_torch[i, j])*torch.log(1-beta_torch[k, l]))


        term_2 = 0
        for i in range(n):
            for l in range(num_blocks):
                term_2 += gamma_posterior_torch[i,l]*torch.log(gamma_posterior_torch[i,l])

        ELBO = term_1 + term_2
        print(f"ELBO: {ELBO}")

        ELBO.backward()
        print("grad_norm:", logits.grad.norm().item())

        with torch.no_grad():
            logits.add_(lr_t * logits.grad)
        logits.grad.zero_()
        gamma_posterior = torch.nn.functional.softmax(logits.detach(), dim=1).cpu().numpy()


        # M step
        gamma_prior = np.average(gamma_posterior, axis=0)
        # gamma_prior = block_sizes/np.sum(block_sizes)

        # beta = probs
        for k in range(num_blocks):
            for l in range(k, num_blocks):
                numerator = np.sum(np.outer(gamma_posterior[:,k],gamma_posterior[:,l]) * (1- np.eye(n)) * A)/2
                denominator = np.sum(np.outer(gamma_posterior[:,k],gamma_posterior[:,l]) * (1- np.eye(n)))/2
                beta[k, l] = numerator/denominator
                beta[l, k] = beta[k, l]

    return gamma_posterior, gamma_prior, beta

In [41]:
seed = 0
# np.random.seed(seed)
# torch.manual_seed(seed)

# Graph construction
block_sizes = [30,70]
num_blocks = len(block_sizes)

n = np.sum(block_sizes)
p = 1.5*n**(-0.3)
r = 0.15*n**(-0.3)
print(f"Intra prob: {p}")
print(f"Inter prob: {r}")

probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

G = nx.stochastic_block_model(block_sizes, probs, seed=seed)
A = nx.to_numpy_array(G)


# Estimate
gamma_posterior, gamma_prior, beta = binary_sbm_estimate_E_step_GD(A, num_blocks, 40, lr = 0.1, mini_batch_p_sample = 0.8)

# NMI
block_labels = np.array([d['block'] for _, d in G.nodes(data=True)])
predicted_labels = np.argmax(gamma_posterior, axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")


Intra prob: 0.376782964726437
Inter prob: 0.0376782964726437
ELBO: -2814.650252285396
grad_norm: 52.972506987009595
ELBO: -2064.343549698502
grad_norm: 2.723325317039129
ELBO: -2081.2167273067744
grad_norm: 2.7737158831695594
ELBO: -2045.593769511317
grad_norm: 2.6190793488379294
ELBO: -2038.916572144861
grad_norm: 2.7109558427395655
ELBO: -2071.636756703682
grad_norm: 2.78717544393921
ELBO: -2022.5051119602451
grad_norm: 2.6971511318160184
ELBO: -2066.321155815896
grad_norm: 2.715024670767362
ELBO: -2061.6163596516694
grad_norm: 2.7705432150369407
ELBO: -2046.4234327328813
grad_norm: 2.7615939217455883
ELBO: -2074.981562568259
grad_norm: 2.8072997734159437
ELBO: -2075.116293018241
grad_norm: 2.838435762829359
ELBO: -2076.9480515325317
grad_norm: 2.7990587622338214
ELBO: -2071.5583581654096
grad_norm: 2.757924323959586
ELBO: -2063.88369719194
grad_norm: 2.854568736590935
ELBO: -2077.951284482759
grad_norm: 2.8497735488681917
ELBO: -2069.1187670863997
grad_norm: 2.8991034193915106
ELBO:

In [413]:
gamma_posterior

array([[0.00824453, 0.99175547],
       [0.00841812, 0.99158188],
       [0.99312603, 0.00687397],
       [0.99364772, 0.00635228],
       [0.00744277, 0.99255723],
       [0.00917948, 0.99082052],
       [0.99460946, 0.00539054],
       [0.99272487, 0.00727513],
       [0.99336613, 0.00663387],
       [0.99166068, 0.00833932],
       [0.99153999, 0.00846001],
       [0.9927163 , 0.0072837 ],
       [0.99396689, 0.00603311],
       [0.99225207, 0.00774793],
       [0.0088546 , 0.9911454 ],
       [0.99320683, 0.00679317],
       [0.99132671, 0.00867329],
       [0.99328073, 0.00671927],
       [0.99163792, 0.00836208],
       [0.99369354, 0.00630646],
       [0.99399453, 0.00600547],
       [0.00731192, 0.99268808],
       [0.99253027, 0.00746973],
       [0.00993315, 0.99006685],
       [0.00683478, 0.99316522],
       [0.99394841, 0.00605159],
       [0.00628127, 0.99371873],
       [0.99094621, 0.00905379],
       [0.99355704, 0.00644296],
       [0.99282094, 0.00717906],
       [0.

In [414]:
beta

array([[0.22535955, 0.21967876],
       [0.21967876, 0.25353357]])

In [410]:
gamma_prior

array([0.97920786, 0.02079214])

In [411]:
block_labels = np.array([d['block'] for _, d in G.nodes(data=True)])
predicted_labels = np.argmax(gamma_posterior, axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")


NMI: 0.020371257409514718


# Poisson SBM

In [26]:
seed = 0
np.random.seed(seed)
pi_true = np.array([0.3, 0.7])
num_blocks = 2

n = 200
p = 3
r = 0.5
print(f"Intra lambda: {p}")
print(f"Inter lambda: {r}")

lambdas = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

z = np.random.choice(num_blocks, size=n, p=pi_true)
A = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(i + 1, n):
        lambda_param = lambdas[z[i], z[j]]
        A[i, j] = np.random.poisson(lam=lambda_param)
        A[j, i] = A[i, j]


Intra lambda: 3
Inter lambda: 0.5


In [27]:
def poisson_sbm_estimate(A, num_blocks, iter):
    
    # Initialize parameters
    gamma_prior = np.random.rand(num_blocks)
    gamma_prior = gamma_prior / np.sum(gamma_prior)
    
    lambda_param = np.random.poisson(lam=int(np.average(A)), size = (2,2)).astype(float)

    gamma_posterior = np.random.rand(n, num_blocks)
    gamma_posterior = gamma_posterior / np.sum(gamma_posterior, axis=1, keepdims=True)

    
    print(gamma_prior)
    
    # EM
    for _ in range(iter):
        # E step
        log_gamma = np.zeros_like(gamma_posterior)
        for i in range(n):
            for k in range(num_blocks):
                term_1 = np.log(gamma_prior[k])
                term_2 = 0
                for j in range(n):
                    if j != i:
                        term_2 += np.sum(gamma_posterior[j, :]*(A[i, j]*np.log(lambda_param[k, :]) - lambda_param[k, :]))
                log_gamma[i, k] = term_1 + term_2

            log_gamma[i, :] = log_gamma[i, :] - logsumexp(log_gamma[i, :])
            gamma_posterior[i, :] = np.exp(log_gamma[i, :])

        # M step
        gamma_prior = np.average(gamma_posterior, axis=0)

        for k in range(num_blocks):
            for l in range(k, num_blocks):
                numerator = np.sum(np.outer(gamma_posterior[:,k], gamma_posterior[:,l]) * (1- np.eye(n)) * A)/2
                denominator = np.sum(np.outer(gamma_posterior[:,k], gamma_posterior[:,l]) * (1- np.eye(n)))/2
                lambda_param[k, l] = numerator/denominator
                lambda_param[l, k] = lambda_param[k, l]

        print(gamma_prior)

    return gamma_posterior


In [28]:
np.random.seed(0)
gamma_posterior = poisson_sbm_estimate(A, num_blocks, 20)

[0.43418691 0.56581309]
[0.28854617 0.71145383]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]
[0.305 0.695]


In [ ]:
block_labels = z
predicted_labels = np.argmax(gamma_posterior, axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")


NMI: 1.0
